In [1]:
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
import json
from tqdm import tqdm

In [2]:
# ==========================================
# 1. SETUP PATH & CONFIGURATION
# ==========================================
INPUT_FILE = "/kaggle/input/datasets/miskiyah/list-scrape/data_primer.csv"
OUTPUT_FILE = "data_primer_shadow_labeled.csv"

# Path model yang benar (berdasarkan struktur folder yang kamu kirim)
MODEL_SENTIMENT_DIR = "/kaggle/input/datasets/miskiyah/list-scrape/model/model/best_s1_sentiment"
MODEL_EMOTION_DIR = "/kaggle/input/datasets/miskiyah/list-scrape/model/model/best_s1_emotion"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# Load Data
df_baru = pd.read_csv(INPUT_FILE)
ulasan_list = df_baru['Review_Clean'].fillna("").astype(str).tolist()

class TokopediaInferenceDataset(Dataset):
    def __init__(self, texts): self.texts = texts
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return str(self.texts[idx])

dataloader_inf = DataLoader(TokopediaInferenceDataset(ulasan_list), batch_size=BATCH_SIZE, shuffle=False)

In [3]:
def run_shadow_labeling(model_dir, task_name):
    print(f"\n--- Memproses Task: {task_name} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    model.eval()

    config_path = os.path.join(model_dir, "config.json")
    with open(config_path, "r") as f:
        config_data = json.load(f)
    
    id2label = config_data.get("id2label", {})
    
    # Cek apakah id2label kosong atau hanya berisi default HuggingFace (LABEL_0, dll)
    is_dummy = True
    if isinstance(id2label, dict) and id2label:
        is_dummy = all(str(v).startswith("LABEL") for v in id2label.values())
        
    # LOGIKA AMAN: Pastikan kita punya label yang bermakna
    if is_dummy or not isinstance(id2label, dict):
        info_path = os.path.join(model_dir, "best_model_info.json")
        labels_list = None
        
        if os.path.exists(info_path):
            with open(info_path, "r") as f:
                info_data = json.load(f)
            labels_list = info_data.get("label_names") or info_data.get("labels")
            
        if labels_list:
            id2label = {i: label for i, label in enumerate(labels_list)}
        else:
            print(f"[!] Warning: Mapping tidak ditemukan! Menggunakan Hardcode {task_name}")
            if task_name == "Sentiment":
                id2label = {0: "Negative", 1: "Positive"}
            else:
                id2label = {0: "Anger", 1: "Fear", 2: "Happy", 3: "Love", 4: "Sadness"}
    
    # KONVERSI AMAN: Pastikan key adalah integer
    final_id2label = {}
    for k, v in id2label.items():
        final_id2label[int(k)] = v
    # ----------------------------------------------

    labels, scores = [], []
    with torch.no_grad():
        for batch_texts in tqdm(dataloader_inf, desc=f"Predicting {task_name}"):
            encodings = tokenizer(batch_texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(DEVICE)
            outputs = model(**encodings)
            probabilities = F.softmax(outputs.logits, dim=-1)
            top_confidences, top_indices = torch.max(probabilities, dim=-1)
            
            # Panggil final_id2label yang sudah pasti dictionary
            labels.extend([final_id2label[idx.item()] for idx in top_indices])
            scores.extend(top_confidences.cpu().numpy())
            
    del model 
    torch.cuda.empty_cache()
    return labels, scores

In [4]:
# ==========================================
# 3. EKSEKUSI & SAVE (SEMUA DATA)
# ==========================================
# Labeling Sentiment
df_baru['shadow_sentiment'], df_baru['confidence_sentiment'] = run_shadow_labeling(MODEL_SENTIMENT_DIR, "Sentiment")

# Labeling Emotion
df_baru['shadow_emotion'], df_baru['confidence_emotion'] = run_shadow_labeling(MODEL_EMOTION_DIR, "Emotion")

# Simpan semua ke CSV
df_baru.to_csv(OUTPUT_FILE, index=False)


--- Memproses Task: Sentiment ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicting Sentiment: 100%|██████████| 1250/1250 [03:47<00:00,  5.48it/s]



--- Memproses Task: Emotion ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicting Emotion: 100%|██████████| 1250/1250 [03:58<00:00,  5.24it/s]


In [5]:
print(f"\n[V] Sukses! Semua data ({len(df_baru)} baris) telah dilabeli.")
print(f"Data lengkap dengan confidence score tersimpan di: {OUTPUT_FILE}")

# Preview untuk kamu cek langsung
print("\n--- Contoh Hasil: ---")
display(df_baru[['Review_Clean', 'shadow_sentiment', 'confidence_sentiment', 'shadow_emotion', 'confidence_emotion']].head())


[V] Sukses! Semua data (40000 baris) telah dilabeli.
Data lengkap dengan confidence score tersimpan di: data_primer_shadow_labeled.csv

--- Contoh Hasil: ---


,Review_Clean,shadow_sentiment,confidence_sentiment,shadow_emotion,confidence_emotion
0,perdana beli oli MOTUL belum di coba semoga be...,Positive,0.998196,Happy,0.782726
1,Toko langganan Kantor. Harga tdk terlalu mahal...,Positive,0.999448,Happy,0.812548
2,"Seller fast respon, kualitasnya bagus juga unt...",Positive,0.999867,Happy,0.962622
3,Maaf nih yah Udh berusaha aku buat nekan kaca ...,Negative,0.986085,Fear,0.388499
4,barang sesuai pesanan dan pengiriman cepat. se...,Positive,0.999867,Happy,0.981221


In [6]:
import pandas as pd

df = pd.read_csv("data_primer_shadow_labeled.csv")

def buat_tabel_persebaran(df, kolom_label):
    # Hitung jumlah dan persentase
    tabel = df[kolom_label].value_counts().reset_index()
    tabel.columns = [kolom_label, 'Jumlah']
    tabel['Persentase (%)'] = (tabel['Jumlah'] / tabel['Jumlah'].sum() * 100).round(2)
    return tabel

# 1. Tabel Distribusi Sentimen
print("=== PERSEBARAN LABEL SENTIMEN ===")
tabel_sentimen = buat_tabel_persebaran(df, 'shadow_sentiment')
display(tabel_sentimen)

# 2. Tabel Distribusi Emosi
print("\n=== PERSEBARAN LABEL EMOSI ===")
tabel_emosi = buat_tabel_persebaran(df, 'shadow_emotion')
display(tabel_emosi)

# 3. Ringkasan Confidence Score (Biar tau seberapa yakin modelmu)
print("\n=== RATA-RATA CONFIDENCE SCORE ===")
ringkasan_conf = {
    "Rata-rata Sentiment Confidence": df['confidence_sentiment'].mean(),
    "Rata-rata Emotion Confidence": df['confidence_emotion'].mean()
}
display(pd.DataFrame([ringkasan_conf]))

=== PERSEBARAN LABEL SENTIMEN ===


,shadow_sentiment,Jumlah,Persentase (%)
0,Positive,23247,58.12
1,Negative,16753,41.88



=== PERSEBARAN LABEL EMOSI ===


,shadow_emotion,Jumlah,Persentase (%)
0,Happy,18248,45.62
1,Fear,6660,16.65
2,Love,6419,16.05
3,Sadness,5101,12.75
4,Anger,3572,8.93



=== RATA-RATA CONFIDENCE SCORE ===


,Rata-rata Sentiment Confidence,Rata-rata Emotion Confidence
0,0.977511,0.682669


In [11]:
import pandas as pd

# 1. LOAD DATA
INPUT_FILE = "data_primer_shadow_labeled.csv"
OUTPUT_FILE = "data_primer_final_annotated.csv"

df = pd.read_csv(INPUT_FILE)

# 2. DEFINISIKAN AMBANG BATAS
# Kita cuma mau buang yang labelnya 'Happy' DAN confidence-nya rendah (misal < 0.7)
# Kamu bisa atur angkanya sesuai selera
THRESHOLD_HAPPY = 0.8 

print(f"Total data awal: {len(df)}")

# 3. LOGIKA FILTER
# Kita buat mask/filter:
# Kondisi A: (Label adalah Happy) DAN (Confidence < 0.7) -> Ini yang mau kita BUANG
# Kondisi B: SEMUA data yang BUKAN Happy -> Ini tetap kita simpan
# Kondisi C: (Label adalah Happy) DAN (Confidence >= 0.7) -> Ini tetap kita simpan

# Cara gampangnya: kita ambil data yang TIDAK memenuhi kondisi A
mask_buang = (df['shadow_emotion'] == 'Happy') & (df['confidence_emotion'] < THRESHOLD_HAPPY)
df_filtered = df[~mask_buang].copy() # Tanda ~ artinya "bukan/not"

# 4. CEK HASIL
print(f"Total data setelah dibuang: {len(df_filtered)}")
print(f"Data 'Happy' yang terbuang: {len(df) - len(df_filtered)}")

# 5. SIMPAN
df_filtered.to_csv(OUTPUT_FILE, index=False)
print(f"File bersih disimpan di: {OUTPUT_FILE}")

Total data awal: 40000
Total data setelah dibuang: 32721
Data 'Happy' yang terbuang: 7279
File bersih disimpan di: data_primer_final_annotated.csv


In [12]:
import pandas as pd

df = pd.read_csv("data_primer_final_annotated.csv")

def buat_tabel_persebaran(df, kolom_label):
    # Hitung jumlah dan persentase
    tabel = df[kolom_label].value_counts().reset_index()
    tabel.columns = [kolom_label, 'Jumlah']
    tabel['Persentase (%)'] = (tabel['Jumlah'] / tabel['Jumlah'].sum() * 100).round(2)
    return tabel

# 1. Tabel Distribusi Sentimen
print("=== PERSEBARAN LABEL SENTIMEN ===")
tabel_sentimen = buat_tabel_persebaran(df, 'shadow_sentiment')
display(tabel_sentimen)

# 2. Tabel Distribusi Emosi
print("\n=== PERSEBARAN LABEL EMOSI ===")
tabel_emosi = buat_tabel_persebaran(df, 'shadow_emotion')
display(tabel_emosi)

# 3. Ringkasan Confidence Score (Biar tau seberapa yakin modelmu)
print("\n=== RATA-RATA CONFIDENCE SCORE ===")
ringkasan_conf = {
    "Rata-rata Sentiment Confidence": df['confidence_sentiment'].mean(),
    "Rata-rata Emotion Confidence": df['confidence_emotion'].mean()
}
display(pd.DataFrame([ringkasan_conf]))

=== PERSEBARAN LABEL SENTIMEN ===


,shadow_sentiment,Jumlah,Persentase (%)
0,Positive,17267,52.77
1,Negative,15454,47.23



=== PERSEBARAN LABEL EMOSI ===


,shadow_emotion,Jumlah,Persentase (%)
0,Happy,10969,33.52
1,Fear,6660,20.35
2,Love,6419,19.62
3,Sadness,5101,15.59
4,Anger,3572,10.92



=== RATA-RATA CONFIDENCE SCORE ===


,Rata-rata Sentiment Confidence,Rata-rata Emotion Confidence
0,0.983651,0.696897
